### 데이터 준비
1. CSV 파일에서 텍스트와 레이블을 읽음
    * TEXT_COL은 텍스트, LABEL_COL은 감정 레이블(0~5 같은 정수)을 의미함
2. 학습, 검증, 테스트 데이터셋으로 분리함
    * stratify=labels는 레이블 분포를 유지하며 나누도록 함


### 텍스트를 BoW 벡터로 변환
1. 단어 집합 생성
    * 학습 데이터에서 가장 많이 등장하는 MAX_FEATURES 개의 단어를 선택함
2. 벡터화
    * 각 단어를 등장 빈도 기반의 고정 길이 벡터로 변환함
    * 각 문장이 길이 MAX_FEATURES의 벡터로 변환됨


### 모델 학습용 데이터 인터페이스 통일
1. 전체 샘플 수와 특정 인덱스의 (X, y) 샘플 반환
    * 데이터를 모델이 읽을 수 있는 형태로 감싸는 wrapper
2. TextDataset은 (X, y)를 PyTorch가 다룰 수 있는 객체로 감싸는 래퍼
    * 각 Dataset 객체는 인덱스(idx)로 샘플을 꺼낼 수 있는 구조가 됨
    * len(dataset)으로 샘플 수 확인, dataset[idx]로 (feature, label) 쉽게 접근 가능
3. DataLoader은 Dataset에서 배치를 자동으로 만들어주는 도구
    * 배치 단위로 나눠서 학습, 검증, 테스트 시 한번에 BATCH_SIZE만큼 처리해줌
    * 학습 시 데이터 순서를 섞어서 학습 안정화, 자동 반복 역할
    * train_loader로 학습 시 배치를 섞어서 모델에 공급, val_loader로 검증 시 순서대로 배치를 공급, test_loader로 테스트 시 순서대로 배치를 공급하는 역할
    

### BoW 모델 구조
1. input
    * 입력 벡터 x는 BoW 벡터(MAX_FEATURES 차원)
2. 첫 번째 linear layer
    * 입력 차원: MAX_FEATURES
    * 출력 차원: 26
    * 역할: 단어 빈도 벡터를 256차원의 공간으로 투사
3. ReLu 활성화
    * 비선형성을 부여해서 모델이 단순 선형 조합 이상의 표현을 학습 가능하게 함
4. 두 번째 linear layer
    * 출력 차원: 클래스 개수인 6
    * 역할: 256차원의 특징을 감정 클래스별 점수로 변환
5. output
    * forward 함수의 출력은 softmax를 거치지 않은 logit
    * CrossEntropyLoss는 내부적으로 softmax + log loss를 처리함


### 학습 과정
1. model.train()을 통해 학습 모드 활성화
2. 배치 반복함
    * X는 모델에 넣어 logits를 계산
    * y는 실제 레이블
    * loss 계산은 CrossEntropyLoss를 통해 모델 출력 logits와 실제 레이블 비교하고, softmax+음의 로그 loss 계산
3. loss.backward()를 통해 역전파로 각 파라미터의 gradient 계산
4. Adam optimizer을 통해 gradient를 이용해 파라미터 업데이트
5. 반복하면서 모델은 BoW 벡터와 레이블 간의 관계 학습


### 평가 과정
1. model.eval()을 통해 평가 모드
2. 학습 중이 아니므로 torch.no_grad()를 통해 역전파 계산 안함
    * 메모리 절약, 속도 향상
3. torch.argmax를 통해 가장 큰 점수를 가진 클래스 인덱스를 선택해 모델이 예측한 label 추출
4. 각 클래스별 Precision, Recall, F1-score, Support 출력
    * Precision: TP/(TP+FP), 해당 클래스로 예측한 것 중 실제로 맞는 비율
    * Recall: TP/(RP+FN), 실제 클래스 중 모델이 맞게 찾아낸 비율
    * F1-score: 2*(Precision*Recall)/(Precision+Recall), Precision과 Recall의 조화평균
    * Support: 각 클래스 실제 샘플 수
5. Confusion Matrix 출력
    * 행: 실제 클래스, 열: 예측 클래스
    * 직관적으로 어떤 클래스가 헷갈리는지 확인하는 용
    * 행=렬: 행을 열로 맞춘 개수, 행!=렬: 행을 열로 틀리게 예측한 개수


### 전체 흐름
1. CSV 데이터에서 텍스트와 레이블 추출
2. 학습, 검증, 테스트로 데이터 분리
3. CountVectorizer로 BoW 벡터 생성
4. PyTorch Dataset, DataLoader로 변환
5. BoWClassifier 모델 초기화
6. CrossEntropyLoss + Adam optimizer 준비
7. 각 epoch마다:
    * train_loader로 배치 단위 학습
    * val_loader로 검증 성능 확인
8. 최종 test_loader로 성능 평가


### 최종 성능
1. Overall accuracy: 0.84로 84%
2. Weighted avg f1: 0.84로 전체적으로 안정적
3. 다수 클래스인 0, 1은 정확도가 높음
4. 소수 클래스인 2. 5는 BoW와 단순 MLP(ANN) 한계로 오분류 발생
    * Recall이 낮음
    * BoW의 단어 순서 무시 특성과 데이터 불균형 영향 때문
5. 모델은 빠르고 단순하지만 복잡한 문맥 감정에서는 한계가 존재함

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import torch.optim as optim

# Configurations
DATA_PATH = "dataset/emotion_recognitions_merged.csv"
TEXT_COL = "text"
LABEL_COL = "label"
EPOCHS = 10
BATCH_SIZE = 64
LR = 1e-3
MAX_FEATURES = 1000

# Dataset Class
class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# BoW Classifier Model
class BoWClassifier(nn.Module):
    def __init__(self, input_dim, num_classes=6):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 256)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(256, num_classes)
    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

# train function
def train_model(model, loader, criterion, optimizer):
    model.train()
    for X, y in loader:
        optimizer.zero_grad()
        outputs = model(X.float())
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

# evaluate function
def evaluate(model, loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for X, y in loader:
            out = model(X.float())
            pred = torch.argmax(out, dim=1)
            preds.extend(pred.cpu().numpy())
            trues.extend(y.cpu().numpy())
    print(classification_report(trues, preds))
    print(confusion_matrix(trues, preds))

if __name__ == "__main__":
    df = pd.read_csv(DATA_PATH)
    texts = df[TEXT_COL].astype(str).tolist()
    labels = df[LABEL_COL].astype(int).tolist()

    X_train, X_temp, y_train, y_temp = train_test_split(texts, labels, test_size=0.3, stratify=labels)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp)

    vectorizer = CountVectorizer(max_features=MAX_FEATURES)
    X_train_bow = vectorizer.fit_transform(X_train).toarray()
    X_val_bow = vectorizer.transform(X_val).toarray()
    X_test_bow = vectorizer.transform(X_test).toarray()

    train_ds = TextDataset(torch.tensor(X_train_bow), torch.tensor(y_train))
    val_ds = TextDataset(torch.tensor(X_val_bow), torch.tensor(y_val))
    test_ds = TextDataset(torch.tensor(X_test_bow), torch.tensor(y_test))

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

    model = BoWClassifier(input_dim=MAX_FEATURES)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR)

    for epoch in range(EPOCHS):
        train_model(model, train_loader, criterion, optimizer)
        print(f"Epoch {epoch+1} validation:")
        evaluate(model, val_loader)

    print("Final Test Performance:")
    evaluate(model, test_loader)


Epoch 1 validation:
              precision    recall  f1-score   support

           0       0.88      0.92      0.90     19047
           1       0.86      0.92      0.89     22174
           2       0.89      0.63      0.74      5429
           3       0.86      0.79      0.82      9004
           4       0.84      0.77      0.80      7513
           5       0.69      0.82      0.75      2354

    accuracy                           0.86     65521
   macro avg       0.84      0.81      0.82     65521
weighted avg       0.86      0.86      0.85     65521

[[17523   701    48   443   270    62]
 [  718 20406   295   321   200   234]
 [  227  1621  3430    84    44    23]
 [  883   563    43  7087   414    14]
 [  603   357    16   242  5766   529]
 [   69   154     6    24   174  1927]]
Epoch 2 validation:
              precision    recall  f1-score   support

           0       0.92      0.88      0.90     19047
           1       0.85      0.93      0.89     22174
           2       

### 클래스 가중치 적용
1. 학습 데이터에서 각 클래스의 샘플 수에 반비례하는 가중치를 계산하도록 함
    * 클래스가 적으면 가중치 증가, 클래스가 많으면 가중치 감소
    * 불균형 데이터셋에서 소수 클래스의 loss를 더 크게 계산하도록 함
2. 기존 CrossEntropyLoss에 weight를 넣으면 손실 계산 시 클래스별 가중치 반영
    * 모델 학습 시 소수 클래스에 더 민감하게 학습하게 됨


### 전체 성능
1. Accuracy: 0.84
2. Macro F1-score: 0.8
    * 클래스별 F1-score 평균으로, 클래스 불균형 영향을 덜 받음
3. weighted F1-score: 0.84
    * 클래스 비중에 따라 평균
4. 소수 클래스 2, 5의 recall이 개선됨
    * precision은 다소 낮지만, recall 우선으로 개선한 결과라 클래스 불균형 대응은 개선되었다고 볼 수 있음

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import torch.optim as optim

# Configurations
DATA_PATH = "dataset/emotion_recognitions_merged.csv"
TEXT_COL = "text"
LABEL_COL = "label"
EPOCHS = 10
BATCH_SIZE = 64
LR = 1e-3
MAX_FEATURES = 1000

# Dataset Class
class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# BoW Classifier Model
class BoWClassifier(nn.Module):
    def __init__(self, input_dim, num_classes=6):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 256)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(256, num_classes)
    def forward(self, x):
        return self.fc2(self.relu(self.fc1(x)))

# Train & Evaluate
def train_model(model, loader, criterion, optimizer):
    model.train()
    for X, y in loader:
        optimizer.zero_grad()
        outputs = model(X.float())
        loss = criterion(outputs, y)
        loss.backward()
        optimizer.step()

def evaluate(model, loader):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for X, y in loader:
            out = model(X.float())
            pred = torch.argmax(out, dim=1)
            preds.extend(pred.cpu().numpy())
            trues.extend(y.cpu().numpy())
    print(classification_report(trues, preds))
    print(confusion_matrix(trues, preds))

# Main
if __name__ == "__main__":
    df = pd.read_csv(DATA_PATH)
    texts = df[TEXT_COL].astype(str).tolist()
    labels = df[LABEL_COL].astype(int).tolist()

    X_train, X_temp, y_train, y_temp = train_test_split(
        texts, labels, test_size=0.3, stratify=labels, random_state=42
    )
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
    )

    vectorizer = CountVectorizer(max_features=MAX_FEATURES)
    X_train_bow = vectorizer.fit_transform(X_train).toarray()
    X_val_bow = vectorizer.transform(X_val).toarray()
    X_test_bow = vectorizer.transform(X_test).toarray()

    train_ds = TextDataset(torch.tensor(X_train_bow), torch.tensor(y_train))
    val_ds = TextDataset(torch.tensor(X_val_bow), torch.tensor(y_val))
    test_ds = TextDataset(torch.tensor(X_test_bow), torch.tensor(y_test))

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

    # Model, Loss Function with Class Weights, Optimizer
    model = BoWClassifier(input_dim=MAX_FEATURES)
    classes = np.unique(y_train)
    class_weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)
    class_weights = torch.tensor(class_weights, dtype=torch.float)
    
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.Adam(model.parameters(), lr=LR)

    for epoch in range(EPOCHS):
        train_model(model, train_loader, criterion, optimizer)
        print(f"\nEpoch {epoch+1} Validation:")
        evaluate(model, val_loader)

    print("\nFinal Test Performance:")
    evaluate(model, test_loader)



Epoch 1 Validation:
              precision    recall  f1-score   support

           0       0.95      0.84      0.90     19047
           1       0.96      0.81      0.88     22174
           2       0.64      0.92      0.76      5429
           3       0.73      0.87      0.79      9004
           4       0.78      0.79      0.79      7513
           5       0.59      0.94      0.73      2354

    accuracy                           0.84     65521
   macro avg       0.78      0.86      0.81     65521
weighted avg       0.86      0.84      0.85     65521

[[16090   391   351  1307   684   224]
 [  382 17973  2113   903   372   431]
 [   54   108  4991   198    45    33]
 [  201   166   200  7821   526    90]
 [  135   135   107   459  5953   724]
 [   11    27    17    60    36  2203]]

Epoch 2 Validation:
              precision    recall  f1-score   support

           0       0.95      0.86      0.90     19047
           1       0.95      0.82      0.88     22174
           2     